# Neutron imaging: division by the open beam image

This notebook goes with Question 10 of the simulation quiz *Bragg Edge Imaging on Viking Sword*.

A radiogram contains the features of the sample, but also the features of the beam itself (guides, choppers, pinhole...). These can be seen in a flat field image: an image of the neutron beam with no sample in it (open beam or "background"). Dividing the image of the object by the open beam image removes the beam features and gives the **transmission** of the sample:

$$T(x,y) = \frac{I_{\mathrm{sample}}(x,y)}{I_{\mathrm{open}}(x,y)}$$

**How to use it**

1. Run the Sword_ODIN simulation twice with the same parameters: once with `Sample=1` and once with `Sample=0`.
2. From each simulation folder (`Sword_ODIN_{date}_{time}`), copy the file `absorption_picture.dat` into a folder called `my_data` next to this notebook. Rename the one with the sample to `data.dat` and the one without the sample to `background.dat`.
3. Run all the cells of this notebook.

If `my_data` is not found, the notebook uses the example files in `example_data` (simulated at 4.3 Å, chopper_mode=3, 1E7 neutron rays).

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt


def read_mccode_2d(path):
    """Read a 2D McStas monitor file (McCode text format): returns intensity, error, counts and extent [cm]."""
    header, rows = {"params": []}, []
    with open(path) as f:
        for line in f:
            if line.startswith("#"):
                key, _, value = line[1:].partition(":")
                if key.strip() == "Param":
                    header["params"].append(value.strip())
                header.setdefault(key.strip(), value.strip())
            elif line.strip():
                rows.append([float(v) for v in line.split()])
    ny = int(header["type"].split("(")[1].split(",")[1].rstrip(")"))
    data = np.array(rows)
    intensity, error, counts = data[:ny], data[ny:2 * ny], data[2 * ny:3 * ny]
    extent = [float(v) for v in header["xylimits"].split()]
    return intensity, error, counts, extent, header


folder = "my_data" if os.path.isdir("my_data") else "example_data"
if folder == "my_data":
    data_file, background_file = os.path.join(folder, "data.dat"), os.path.join(folder, "background.dat")
else:
    data_file, background_file = os.path.join(folder, "data_4.3A.dat"), os.path.join(folder, "background_4.3A.dat")
print("Using", data_file, "and", background_file)

data, data_err, data_n, extent, header = read_mccode_2d(data_file)
background, bg_err, bg_n, _, _ = read_mccode_2d(background_file)
print("Simulation parameters:", ", ".join(header["params"]))

## The two raw images

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, img, title in zip(axes, [data, background], ["With the sword (data)", "Open beam (background)"]):
    im = ax.imshow(img, origin="lower", extent=extent, cmap="viridis")
    ax.set_title(title)
    ax.set_xlabel("X position [cm]")
    ax.set_ylabel("Y position [cm]")
    fig.colorbar(im, ax=ax, label="Intensity [n/s]")
plt.tight_layout()
plt.show()

## Transmission image

Pixels where the open beam has no counts are left empty. The colour scale is limited to the central 98 % of the values so that a few noisy pixels do not hide the contrast.

In [ ]:
with np.errstate(divide="ignore", invalid="ignore"):
    transmission = np.where(background > 0, data / background, np.nan)

low, high = np.nanpercentile(transmission, [1, 99])
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(transmission, origin="lower", extent=extent, cmap="viridis", vmin=low, vmax=high)
ax.set_title("Transmission = data / background")
ax.set_xlabel("X position [cm]")
ax.set_ylabel("Y position [cm]")
fig.colorbar(im, ax=ax, label="Transmission")
plt.show()

Compare the transmission image with the raw image: which features have disappeared, and which ones are still visible? Use your observations to answer Question 10 of the quiz.